# Train Document Profile Node2Vec

Notebook nay xay dung goi y tai lieu chi dua tren ho so `user` va `document`.

Cac thuoc tinh duoc dung:
- `major`
- `school`
- `cohort`

Khong dung `interactions` hay `friendships` trong phien ban nay.

In [1]:
from __future__ import annotations

from pathlib import Path
import re
import unicodedata

import pandas as pd
import networkx as nx
from node2vec import Node2Vec
from sklearn.metrics.pairwise import cosine_similarity

In [2]:
DATA_DIR = Path.cwd()
USERS_FILE = DATA_DIR / 'document_train_users.csv'
DOCUMENTS_FILE = DATA_DIR / 'document_train_documents.csv'

EMBEDDINGS_OUTPUT = DATA_DIR / 'document_profile_node2vec_embeddings.csv'
RECOMMEND_OUTPUT = DATA_DIR / 'document_profile_recommendations_all_users.csv'
VIEW_OUTPUT = DATA_DIR / 'document_profile_recommendations_view.csv'

TOP_K = 20
DIMENSIONS = 64
WALK_LENGTH = 20
NUM_WALKS = 50
WINDOW = 10
MIN_COUNT = 1
WORKERS = 1

In [3]:
users_df = pd.read_csv(USERS_FILE)
documents_df = pd.read_csv(DOCUMENTS_FILE)

print('users    :', len(users_df))
print('documents:', len(documents_df))

users_df.head()

users    : 4121
documents: 152


,userId,email,displayName,interests,school,major,cohort,role,status,profileVisibility
0,00011052-b71e-5d93-a12d-b4e8b200fbdf,fb_3221@facebook-combined.local,Trần Thị Hương,"Quan tâm đến bạn bè, cộng đồng và những hoạt đ...",Đại học Quy Nhơn,Giáo dục tiểu học,K33,USER,ACTIVE,PUBLIC
1,00070f68-8cae-590f-aad0-ce7bf65d54e5,fb_901@facebook-combined.local,Trần Đức Châu,"Quan tâm đến bạn bè, cộng đồng và những hoạt đ...",Đại học Khoa học Xã hội và Nhân văn - ĐHQGHN,Quan hệ công chúng,K38,USER,ACTIVE,PUBLIC
2,001409a7-980b-5ecf-9f54-2a8c2953bf6c,fb_2331@facebook-combined.local,Hồ Gia Hải,"Ưu tiên các mối quan hệ chân thành, giao tiếp ...",Đại học Nông Lâm TP.HCM,Chăn nuôi,K35,USER,ACTIVE,PUBLIC
3,0028d64f-183d-53f5-8017-7713f18eb0da,fb_882@facebook-combined.local,Lê Bảo Châu,"Thích trò chuyện, làm việc nhóm và lan tỏa nhữ...",Đại học Khoa học Xã hội và Nhân văn - ĐHQGHN,Quan hệ công chúng,K41,USER,ACTIVE,PUBLIC
4,0031cb49-1320-525a-8e72-3abf339f05ea,fb_430@facebook-combined.local,Đỗ Quang Bình,"Yêu thích môi trường cởi mở, nơi mọi người có ...",Học viện Báo chí và Tuyên truyền,Truyền thông đa phương tiện,K34,USER,ACTIVE,PUBLIC


In [4]:
documents_df.head()

,documentId,title,fileName,fileType,subject,school,major,cohort,description,tags,visibility,status,viewsCount,downloadsCount,uploaderId,createdAt,updatedAt,sourceType
0,7bb9d9b8-c63a-4c92-a097-23a236e5120f,K-Means From Theory to Practice (3),K-Means From Theory to Practice (3).pdf,PDF,ML,ĐH Quy Nhơn,CNTT,K45,NaN,NaN,PUBLIC,ACTIVE,1,0,2901fc71-a8a0-4634-a6af-6a0d980430d7,2026-04-27T07:56:33.611Z,2026-05-11T07:02:58.935Z,NaN
1,56ebfdfe-13dc-473c-bdea-ab78970246f8,Báo cáo thực tập-LeXuanNgoc,BÃ¡o cÃ¡o thá»±c táº­p-LeXuanNgoc.doc,DOC,ML,ĐH Quy Nhơn,CNTT,K45,NaN,NaN,PUBLIC,ACTIVE,1,0,2901fc71-a8a0-4634-a6af-6a0d980430d7,2026-05-11T07:02:45.309Z,2026-05-11T07:16:23.513Z,NaN
2,1756f62b-88c5-41f8-aa54-a21bac2a86a3,Thủy sản căn bản,Thủy sản căn bản.docx,DOC,Nông nghiệp,Học viện Báo chí và Tuyên truyền,Chăn nuôi,K30,"Tài liệu học tập về thủy sản căn bản, hỗ trợ s...",thủy sản|nông nghiệp|chăn nuôi|hữu cơ,PUBLIC,ACTIVE,0,0,8792b584-b835-4340-889b-06f1ae962e99,2026-05-16T08:31:19.020Z,2026-05-16T08:31:19.020Z,NaN
3,35976ac5-1a40-4616-a15b-b6dbd7737262,Biên dịch văn bản chuyên ngành,Biên dịch văn bản chuyên ngành.pptx,PPT,Ngoại ngữ,Học viện Công nghệ Bưu chính Viễn thông,Biên phiên dịch,K33,Tài liệu học tập về biên dịch văn bản chuyên n...,học thuật|ngoại ngữ|biên dịch|giao tiếp,PUBLIC,ACTIVE,0,0,7874ff7c-813c-5627-b476-244d84950793,2026-05-16T08:31:20.901Z,2026-05-16T08:31:20.901Z,NaN
4,ecb0c11d-7693-4815-b441-e1ebe2b96b3d,Luật lao động,Luật lao động.pptx,PPT,Luật,Học viện Ngân hàng,Luật dân sự,K41,"Tài liệu học tập về luật lao động, hỗ trợ sinh...",lao động|dân sự|hợp đồng|luật,PUBLIC,ACTIVE,0,0,64353153-04ad-5366-8bb8-22e1d3ce53d4,2026-05-16T08:31:21.857Z,2026-05-16T08:31:21.857Z,NaN


In [5]:
def normalize_text(value: object) -> str:
    if pd.isna(value):
        return ''

    text = str(value).strip().lower()
    text = unicodedata.normalize('NFKC', text)
    text = ''.join(
        char for char in unicodedata.normalize('NFD', text)
        if unicodedata.category(char) != 'Mn'
    )
    text = re.sub(r'[^a-z0-9]+', '_', text)
    return text.strip('_')


def build_attribute_node_id(prefix: str, value: object) -> str | None:
    normalized = normalize_text(value)
    if not normalized:
        return None
    return f'{prefix}_{normalized}'

In [6]:
profile_users_df = users_df[['userId', 'displayName', 'school', 'major', 'cohort']].copy()
profile_documents_df = documents_df[['documentId', 'title', 'school', 'major', 'cohort']].copy()

for field in ['school', 'major', 'cohort']:
    profile_users_df[f'norm_{field}'] = profile_users_df[field].map(normalize_text)
    profile_documents_df[f'norm_{field}'] = profile_documents_df[field].map(normalize_text)

profile_users_df.head()

,userId,displayName,school,major,cohort,norm_school,norm_major,norm_cohort
0,00011052-b71e-5d93-a12d-b4e8b200fbdf,Trần Thị Hương,Đại học Quy Nhơn,Giáo dục tiểu học,K33,ai_hoc_quy_nhon,giao_duc_tieu_hoc,k33
1,00070f68-8cae-590f-aad0-ce7bf65d54e5,Trần Đức Châu,Đại học Khoa học Xã hội và Nhân văn - ĐHQGHN,Quan hệ công chúng,K38,ai_hoc_khoa_hoc_xa_hoi_va_nhan_van_hqghn,quan_he_cong_chung,k38
2,001409a7-980b-5ecf-9f54-2a8c2953bf6c,Hồ Gia Hải,Đại học Nông Lâm TP.HCM,Chăn nuôi,K35,ai_hoc_nong_lam_tp_hcm,chan_nuoi,k35
3,0028d64f-183d-53f5-8017-7713f18eb0da,Lê Bảo Châu,Đại học Khoa học Xã hội và Nhân văn - ĐHQGHN,Quan hệ công chúng,K41,ai_hoc_khoa_hoc_xa_hoi_va_nhan_van_hqghn,quan_he_cong_chung,k41
4,0031cb49-1320-525a-8e72-3abf339f05ea,Đỗ Quang Bình,Học viện Báo chí và Tuyên truyền,Truyền thông đa phương tiện,K34,hoc_vien_bao_chi_va_tuyen_truyen,truyen_thong_a_phuong_tien,k34


In [7]:
profile_documents_df.head()

,documentId,title,school,major,cohort,norm_school,norm_major,norm_cohort
0,7bb9d9b8-c63a-4c92-a097-23a236e5120f,K-Means From Theory to Practice (3),ĐH Quy Nhơn,CNTT,K45,h_quy_nhon,cntt,k45
1,56ebfdfe-13dc-473c-bdea-ab78970246f8,Báo cáo thực tập-LeXuanNgoc,ĐH Quy Nhơn,CNTT,K45,h_quy_nhon,cntt,k45
2,1756f62b-88c5-41f8-aa54-a21bac2a86a3,Thủy sản căn bản,Học viện Báo chí và Tuyên truyền,Chăn nuôi,K30,hoc_vien_bao_chi_va_tuyen_truyen,chan_nuoi,k30
3,35976ac5-1a40-4616-a15b-b6dbd7737262,Biên dịch văn bản chuyên ngành,Học viện Công nghệ Bưu chính Viễn thông,Biên phiên dịch,K33,hoc_vien_cong_nghe_buu_chinh_vien_thong,bien_phien_dich,k33
4,ecb0c11d-7693-4815-b441-e1ebe2b96b3d,Luật lao động,Học viện Ngân hàng,Luật dân sự,K41,hoc_vien_ngan_hang,luat_dan_su,k41


In [8]:
def build_profile_graph(users_df: pd.DataFrame, documents_df: pd.DataFrame) -> nx.Graph:
    graph = nx.Graph()

    for row in users_df.itertuples(index=False):
        user_node_id = f'U_{row.userId}'
        graph.add_node(
            user_node_id,
            nodeType='USER',
            entityId=str(row.userId),
            label=row.displayName if pd.notna(row.displayName) else str(row.userId),
        )

        attribute_specs = [
            ('M', 'MAJOR', row.major, 2.0, 'HAS_MAJOR'),
            ('S', 'SCHOOL', row.school, 1.0, 'STUDIES_AT'),
            ('C', 'COHORT', row.cohort, 1.0, 'BELONGS_TO_COHORT'),
        ]

        for prefix, node_type, raw_value, weight, edge_type in attribute_specs:
            attribute_node_id = build_attribute_node_id(prefix, raw_value)
            if not attribute_node_id:
                continue
            graph.add_node(
                attribute_node_id,
                nodeType=node_type,
                entityId=normalize_text(raw_value),
                label=str(raw_value),
            )
            graph.add_edge(user_node_id, attribute_node_id, weight=weight, edgeType=edge_type)

    for row in documents_df.itertuples(index=False):
        document_node_id = f'D_{row.documentId}'
        graph.add_node(
            document_node_id,
            nodeType='DOCUMENT',
            entityId=str(row.documentId),
            label=row.title if pd.notna(row.title) else str(row.documentId),
        )

        attribute_specs = [
            ('M', 'MAJOR', row.major, 2.0, 'FOR_MAJOR'),
            ('S', 'SCHOOL', row.school, 1.0, 'FROM_SCHOOL'),
            ('C', 'COHORT', row.cohort, 1.0, 'FOR_COHORT'),
        ]

        for prefix, node_type, raw_value, weight, edge_type in attribute_specs:
            attribute_node_id = build_attribute_node_id(prefix, raw_value)
            if not attribute_node_id:
                continue
            graph.add_node(
                attribute_node_id,
                nodeType=node_type,
                entityId=normalize_text(raw_value),
                label=str(raw_value),
            )
            graph.add_edge(document_node_id, attribute_node_id, weight=weight, edgeType=edge_type)

    return graph

In [9]:
graph = build_profile_graph(profile_users_df, profile_documents_df)

print('graph nodes:', graph.number_of_nodes())
print('graph edges:', graph.number_of_edges())

node_types = pd.Series([attrs['nodeType'] for _, attrs in graph.nodes(data=True)]).value_counts()
edge_types = pd.Series([attrs['edgeType'] for _, _, attrs in graph.edges(data=True)]).value_counts()

display(node_types.rename_axis('nodeType').reset_index(name='count'))
display(edge_types.rename_axis('edgeType').reset_index(name='count'))

graph nodes: 4375
graph edges: 12815


,nodeType,count
0,USER,4121
1,DOCUMENT,152
2,MAJOR,45
3,SCHOOL,41
4,COHORT,16


,edgeType,count
0,HAS_MAJOR,4121
1,STUDIES_AT,4121
2,BELONGS_TO_COHORT,4121
3,FROM_SCHOOL,152
4,FOR_COHORT,152
5,FOR_MAJOR,148


In [10]:
node2vec = Node2Vec(
    graph,
    dimensions=DIMENSIONS,
    walk_length=WALK_LENGTH,
    num_walks=NUM_WALKS,
    workers=WORKERS,
    weight_key='weight',
)
model = node2vec.fit(window=WINDOW, min_count=MIN_COUNT, batch_words=128)

Computing transition probabilities:   0%|          | 0/4375 [00:00<?, ?it/s]

Generating walks (CPU: 1): 100%|██████████| 50/50 [00:53<00:00,  1.08s/it]


In [11]:
rows = []
for node_id, attrs in graph.nodes(data=True):
    vector = model.wv[node_id]
    row = {
        'nodeId': node_id,
        'nodeType': attrs.get('nodeType', ''),
        'entityId': attrs.get('entityId', ''),
        'label': attrs.get('label', ''),
    }
    for index, value in enumerate(vector):
        row[f'emb_{index}'] = float(value)
    rows.append(row)

embeddings_df = pd.DataFrame(rows)
embeddings_df.to_csv(EMBEDDINGS_OUTPUT, index=False, encoding='utf-8')
embeddings_df.head()

,nodeId,nodeType,entityId,label,emb_0,emb_1,emb_2,emb_3,emb_4,emb_5,...,emb_54,emb_55,emb_56,emb_57,emb_58,emb_59,emb_60,emb_61,emb_62,emb_63
0,U_00011052-b71e-5d93-a12d-b4e8b200fbdf,USER,00011052-b71e-5d93-a12d-b4e8b200fbdf,Trần Thị Hương,0.495882,-0.389749,-0.509495,0.124311,0.569710,0.121316,...,0.472364,0.121505,-0.200447,0.054390,-0.880404,0.004971,0.080250,-0.481014,0.009681,0.015201
1,M_giao_duc_tieu_hoc,MAJOR,giao_duc_tieu_hoc,Giáo dục tiểu học,0.367742,-0.540396,-0.444883,-0.010421,0.765508,0.279055,...,0.217819,0.155120,0.181218,0.029252,-0.408796,0.368666,0.096896,-0.259487,0.292805,-0.058620
2,S_ai_hoc_quy_nhon,SCHOOL,ai_hoc_quy_nhon,Đại học Quy Nhơn,0.411524,-0.545956,-0.462313,0.090103,0.819462,0.316004,...,0.128866,0.215114,0.248975,-0.004812,-0.408969,0.233242,0.076834,-0.242148,0.244522,0.013161
3,C_k33,COHORT,k33,K33,0.040317,0.242776,0.118028,0.490596,0.102745,-0.309737,...,0.636454,0.021041,-0.208419,-0.359594,-0.250664,-0.068892,0.094735,-0.340039,-0.453731,-0.321044
4,U_00070f68-8cae-590f-aad0-ce7bf65d54e5,USER,00070f68-8cae-590f-aad0-ce7bf65d54e5,Trần Đức Châu,0.059107,-0.518606,0.424711,-0.327293,0.291791,-0.408228,...,-0.416708,-0.201547,0.113724,0.082789,-0.419798,0.417427,-0.156047,-0.184935,0.161814,-0.460757


In [12]:
def build_recommendations(embeddings_df: pd.DataFrame, top_k: int = TOP_K) -> pd.DataFrame:
    embed_cols = [column for column in embeddings_df.columns if column.startswith('emb_')]
    user_embeddings = embeddings_df[embeddings_df['nodeType'].eq('USER')].copy()
    document_embeddings = embeddings_df[embeddings_df['nodeType'].eq('DOCUMENT')].copy()

    document_vectors = document_embeddings[embed_cols].to_numpy()
    document_ids = document_embeddings['entityId'].astype(str).tolist()

    rows = []
    for user_row in user_embeddings.itertuples(index=False):
        user_id = str(user_row.entityId)
        user_vector = pd.DataFrame([getattr(user_row, column) for column in embed_cols]).T.to_numpy().reshape(1, -1)
        similarities = cosine_similarity(user_vector, document_vectors)[0]

        ranked = sorted(
            zip(document_ids, similarities, strict=False),
            key=lambda item: item[1],
            reverse=True,
        )[:top_k]

        for rank, (document_id, score) in enumerate(ranked, start=1):
            rows.append([user_id, document_id, rank, float(score), 'document_profile_node2vec'])

    return pd.DataFrame(
        rows,
        columns=['userId', 'documentId', 'rank', 'similarityScore', 'recommendationSource'],
    )

In [13]:
recommend_df = build_recommendations(embeddings_df)
recommend_df.to_csv(RECOMMEND_OUTPUT, index=False, encoding='utf-8')
recommend_df.head(20)

,userId,documentId,rank,similarityScore,recommendationSource
0,00011052-b71e-5d93-a12d-b4e8b200fbdf,b419f591-59cf-45b4-82cf-a2f93cc6bfd8,1,0.672998,document_profile_node2vec
1,00011052-b71e-5d93-a12d-b4e8b200fbdf,bb97dd4f-3e16-4947-8e44-eef5bd0dbb5a,2,0.642205,document_profile_node2vec
2,00011052-b71e-5d93-a12d-b4e8b200fbdf,dbf3b273-4584-427a-8c8d-0ebad4b8820f,3,0.538105,document_profile_node2vec
3,00011052-b71e-5d93-a12d-b4e8b200fbdf,7ef67d35-3b6e-48cc-b09c-8a72d129c965,4,0.469729,document_profile_node2vec
4,00011052-b71e-5d93-a12d-b4e8b200fbdf,4d427e67-f6da-4163-8be9-c0f9052006bc,5,0.426148,document_profile_node2vec
5,00011052-b71e-5d93-a12d-b4e8b200fbdf,46214677-b556-43e4-8396-4f307a29a194,6,0.401034,document_profile_node2vec
6,00011052-b71e-5d93-a12d-b4e8b200fbdf,e30480d6-280a-49d4-917a-5b3c8e12ecde,7,0.385643,document_profile_node2vec
7,00011052-b71e-5d93-a12d-b4e8b200fbdf,0bacd07c-c4aa-468b-b090-a32242b08a11,8,0.376177,document_profile_node2vec
8,00011052-b71e-5d93-a12d-b4e8b200fbdf,8d735e5f-e55c-4fc8-b82a-47dc025ddff2,9,0.350375,document_profile_node2vec
9,00011052-b71e-5d93-a12d-b4e8b200fbdf,0c69b65e-1793-4a20-9115-7564e6076a97,10,0.344903,document_profile_node2vec


In [14]:
user_cols = ['userId', 'displayName', 'school', 'major', 'cohort']
doc_cols = ['documentId', 'title', 'school', 'major', 'cohort']

view_df = recommend_df.merge(users_df[user_cols], on='userId', how='left').rename(columns={
    'displayName': 'userDisplayName',
    'school': 'userSchool',
    'major': 'userMajor',
    'cohort': 'userCohort',
})

view_df = view_df.merge(documents_df[doc_cols], on='documentId', how='left').rename(columns={
    'title': 'documentTitle',
    'school': 'documentSchool',
    'major': 'documentMajor',
    'cohort': 'documentCohort',
})

view_df = view_df.sort_values(['userId', 'rank']).reset_index(drop=True)
view_df.to_csv(VIEW_OUTPUT, index=False, encoding='utf-8')
view_df.head(20)

,userId,documentId,rank,similarityScore,recommendationSource,userDisplayName,userSchool,userMajor,userCohort,documentTitle,documentSchool,documentMajor,documentCohort
0,00011052-b71e-5d93-a12d-b4e8b200fbdf,b419f591-59cf-45b4-82cf-a2f93cc6bfd8,1,0.672998,document_profile_node2vec,Trần Thị Hương,Đại học Quy Nhơn,Giáo dục tiểu học,K33,Tâm lý học sức khỏe,Đại học Quy Nhơn,Dược học,K33
1,00011052-b71e-5d93-a12d-b4e8b200fbdf,bb97dd4f-3e16-4947-8e44-eef5bd0dbb5a,2,0.642205,document_profile_node2vec,Trần Thị Hương,Đại học Quy Nhơn,Giáo dục tiểu học,K33,Truyền thông đa phương tiện,Đại học Quy Nhơn,NaN,K43
2,00011052-b71e-5d93-a12d-b4e8b200fbdf,dbf3b273-4584-427a-8c8d-0ebad4b8820f,3,0.538105,document_profile_node2vec,Trần Thị Hương,Đại học Quy Nhơn,Giáo dục tiểu học,K33,Nghiên cứu thị trường,Đại học Quy Nhơn,Digital Marketing,K35
3,00011052-b71e-5d93-a12d-b4e8b200fbdf,7ef67d35-3b6e-48cc-b09c-8a72d129c965,4,0.469729,document_profile_node2vec,Trần Thị Hương,Đại học Quy Nhơn,Giáo dục tiểu học,K33,Kịch bản truyền thông,Đại học Xây dựng Hà Nội,Truyền thông,K33
4,00011052-b71e-5d93-a12d-b4e8b200fbdf,4d427e67-f6da-4163-8be9-c0f9052006bc,5,0.426148,document_profile_node2vec,Trần Thị Hương,Đại học Quy Nhơn,Giáo dục tiểu học,K33,Luật sở hữu trí tuệ,Đại học Luật Hà Nội,Luật dân sự,K33
5,00011052-b71e-5d93-a12d-b4e8b200fbdf,46214677-b556-43e4-8396-4f307a29a194,6,0.401034,document_profile_node2vec,Trần Thị Hương,Đại học Quy Nhơn,Giáo dục tiểu học,K33,Pháp luật về bảo vệ dữ liệu cá nhân,Đại học Bách khoa Hà Nội,Luật kinh tế,K33
6,00011052-b71e-5d93-a12d-b4e8b200fbdf,e30480d6-280a-49d4-917a-5b3c8e12ecde,7,0.385643,document_profile_node2vec,Trần Thị Hương,Đại học Quy Nhơn,Giáo dục tiểu học,K33,Kế toán thuế thực hành,Học viện Ngân hàng,Kiểm toán,K33
7,00011052-b71e-5d93-a12d-b4e8b200fbdf,0bacd07c-c4aa-468b-b090-a32242b08a11,8,0.376177,document_profile_node2vec,Trần Thị Hương,Đại học Quy Nhơn,Giáo dục tiểu học,K33,Thanh toán quốc tế,Đại học Cần Thơ,Tài chính ngân hàng,K33
8,00011052-b71e-5d93-a12d-b4e8b200fbdf,8d735e5f-e55c-4fc8-b82a-47dc025ddff2,9,0.350375,document_profile_node2vec,Trần Thị Hương,Đại học Quy Nhơn,Giáo dục tiểu học,K33,Kiểm soát nội bộ,Học viện Công nghệ Bưu chính Viễn thông,Kiểm toán,K33
9,00011052-b71e-5d93-a12d-b4e8b200fbdf,0c69b65e-1793-4a20-9115-7564e6076a97,10,0.344903,document_profile_node2vec,Trần Thị Hương,Đại học Quy Nhơn,Giáo dục tiểu học,K33,Sản xuất nội dung số,Đại học Kinh tế - Luật,Truyền thông,K44


In [15]:
selected_user_id = '2901fc71-a8a0-4634-a6af-6a0d980430d7'
top_n = 10

user_result = view_df[
    view_df['userId'].astype(str) == str(selected_user_id)
].head(top_n).copy()

if user_result.empty:
    raise ValueError(f'Khong tim thay recommendation cho userId: {selected_user_id}')

display(user_result[[
    'userId',
    'userDisplayName',
    'userSchool',
    'userMajor',
    'userCohort',
    'documentId',
    'documentTitle',
    'documentSchool',
    'documentMajor',
    'documentCohort',
    'similarityScore',
    'recommendationSource',
]])

,userId,userDisplayName,userSchool,userMajor,userCohort,documentId,documentTitle,documentSchool,documentMajor,documentCohort,similarityScore,recommendationSource
12860,2901fc71-a8a0-4634-a6af-6a0d980430d7,Nguyễn Duy Thịnh,Đại học Văn Lang,Digital Marketing,K39,36485519-e7f0-4359-9b8d-aba529586cd8,Digital marketing tổng quan,Đại học Hà Nội,Digital Marketing,K31,0.662516,document_profile_node2vec
12861,2901fc71-a8a0-4634-a6af-6a0d980430d7,Nguyễn Duy Thịnh,Đại học Văn Lang,Digital Marketing,K39,0bff6819-b876-4478-823e-d9b52cb8f5ee,Content marketing cho mạng xã hội,Đại học Công nghệ - ĐHQGHN,Digital Marketing,K42,0.645300,document_profile_node2vec
12862,2901fc71-a8a0-4634-a6af-6a0d980430d7,Nguyễn Duy Thịnh,Đại học Văn Lang,Digital Marketing,K39,43d79757-0aaf-4816-ab6a-457bc89bd175,Phân tích hành vi khách hàng,Đại học Tài nguyên và Môi trường Hà Nội,Digital Marketing,K30,0.612313,document_profile_node2vec
12863,2901fc71-a8a0-4634-a6af-6a0d980430d7,Nguyễn Duy Thịnh,Đại học Văn Lang,Digital Marketing,K39,02152b68-76a8-4dea-be7f-dfabd60d64fc,Truyền thông tích hợp,Đại học Sư phạm TP.HCM,Digital Marketing,K42,0.609462,document_profile_node2vec
12864,2901fc71-a8a0-4634-a6af-6a0d980430d7,Nguyễn Duy Thịnh,Đại học Văn Lang,Digital Marketing,K39,598fa544-3b2d-46b2-9f45-8e39f3e44e86,Quảng cáo trực tuyến,Đại học Luật TP.HCM,Digital Marketing,K34,0.602054,document_profile_node2vec
12865,2901fc71-a8a0-4634-a6af-6a0d980430d7,Nguyễn Duy Thịnh,Đại học Văn Lang,Digital Marketing,K39,41483fa8-bff5-4ab9-90f2-bd50a1f8b325,SEO và tối ưu nội dung,Đại học FPT,Digital Marketing,K44,0.594323,document_profile_node2vec
12866,2901fc71-a8a0-4634-a6af-6a0d980430d7,Nguyễn Duy Thịnh,Đại học Văn Lang,Digital Marketing,K39,dbf3b273-4584-427a-8c8d-0ebad4b8820f,Nghiên cứu thị trường,Đại học Quy Nhơn,Digital Marketing,K35,0.548129,document_profile_node2vec
12867,2901fc71-a8a0-4634-a6af-6a0d980430d7,Nguyễn Duy Thịnh,Đại học Văn Lang,Digital Marketing,K39,b11b08d5-0afe-4036-8f3e-48281e29208b,Quản trị thương hiệu,Đại học Cần Thơ,Digital Marketing,K32,0.540736,document_profile_node2vec
12868,2901fc71-a8a0-4634-a6af-6a0d980430d7,Nguyễn Duy Thịnh,Đại học Văn Lang,Digital Marketing,K39,f8872ab3-f1dc-43a1-ac15-793841c20155,Phát triển ứng dụng web,Đại học Văn Lang,Công nghệ thông tin,K30,0.532186,document_profile_node2vec
12869,2901fc71-a8a0-4634-a6af-6a0d980430d7,Nguyễn Duy Thịnh,Đại học Văn Lang,Digital Marketing,K39,07e9f472-8e83-437d-8b88-c8ccd664758f,Hệ thống tưới tiêu,Đại học Văn Lang,Chăn nuôi,K37,0.531026,document_profile_node2vec


In [16]:
view_df.columns.tolist()

['userId',
 'documentId',
 'rank',
 'similarityScore',
 'recommendationSource',
 'userDisplayName',
 'userSchool',
 'userMajor',
 'userCohort',
 'documentTitle',
 'documentSchool',
 'documentMajor',
 'documentCohort']